<div style="text-align: right">Author: Yuktha Bhadane (yb294@cornell.edu)</div>

### Phase Out Order: Excel Workbook
This notebook creates an excel workbook for the power plant phaseout scenarios noting a unique rank for each asset that is phased out. The rankings are first ordered by year and then by subsector priority i.e. Coal → Oil → Gas. Each sheet in the excel workbook is for the set of eight developing countries (India, Indonesia, South Africa, Mexico, Viet Nam, Iran, Thailand, and Egypt).

In [3]:
import pandas as pd
from pathlib import Path
import os
import re

def get_specified_countries():
    """Return the specified list of country codes."""
    return ['IN', 'ID', 'IR', 'TH', 'VN', 'EG', 'MX', 'ZA']

def create_ranked_phaseout_excel(base_path, output_path):
    # Use the specified countries
    countries = get_specified_countries()
    if not countries:
        print("No countries specified!")
        return
    
    print(f"Processing {len(countries)} specified countries")
    
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for country_code in countries:
            print(f"\nProcessing country: {country_code}")
            
            # Define criteria files in desired order
            criteria_files = {
                'Maturity': f'v2_power_plant_phaseout_order_by_maturity_{country_code}_2050.csv',
                'Emission Factor': f'v2_power_plant_phaseout_order_by_emission_factor_{country_code}_2050.csv',
                'Benefits/Cost/Maturity': f'v2_power_plant_phaseout_order_by_emissions_per_OC_maturity_{country_code}_2050.csv'
            }
            
            # List to store DataFrames
            dfs = []
            
            # Process each criteria file
            for criteria_name, filename in criteria_files.items():
                file_path = Path(base_path) / filename
                if not os.path.exists(file_path):
                    print(f"File not found: {filename}")
                    continue
                
                # Read CSV 
                df = pd.read_csv(file_path)
                
                # Apply the improved ranking method from visualization code
                # Define the order of fuel types for phaseout
                fuel_order = ['Coal', 'Oil', 'Gas']
                
                # Assign ranks based on year, then fuel type, then order in CSV
                plant_ranks = {}
                current_rank = 1
                
                # For each year in chronological order
                for year in sorted(df['year'].unique()):
                    year_df = df[df['year'] == year]
                    
                    # For each fuel type in the specified order
                    for fuel in fuel_order:
                        # Get plants of this fuel type for this year
                        fuel_plants = year_df[year_df['subsector'] == fuel]
                        
                        # Assign ranks to these plants (in CSV order)
                        for asset_id in fuel_plants['uniqueforwardassetid']:
                            if asset_id not in plant_ranks:
                                plant_ranks[asset_id] = current_rank
                                current_rank += 1
                
                # Add rank to dataframe based on the new ranking method
                df['rank'] = df['uniqueforwardassetid'].map(plant_ranks)
                
                # Keep needed columns
                columns_to_keep = ['rank', 'asset_name', 'subsector', 'fraction', 'amount_mtco2', 'year']
                df = df[columns_to_keep]
                
                # Format fractions as percentages
                df['fraction'] = df['fraction'].apply(lambda x: f"{x*100:.1f}%")
                
                # Round emissions to 3 decimal places
                df['amount_mtco2'] = df['amount_mtco2'].round(3)
                
                # Sort by rank
                df = df.sort_values(by='rank')
                
                # Rename columns
                rename_dict = {col: f"{col} ({criteria_name})" for col in columns_to_keep}
                df = df.rename(columns=rename_dict)
                
                # Add narrow empty column after each criteria (except the last one)
                if criteria_name != 'Benefits/Cost/Maturity':
                    df['_'] = ''  # Using underscore for empty column name
                
                dfs.append(df)
            
            # Combine all DataFrames
            if dfs:  # Only proceed if we have data
                result_df = pd.concat(dfs, axis=1)
                
                # Write to Excel
                sheet_name = f"{country_code}"
                result_df.to_excel(writer, sheet_name=sheet_name, index=False)
                
                # Adjust column widths
                worksheet = writer.sheets[sheet_name]
                for idx, col in enumerate(result_df.columns):
                    # Set narrow width for separator columns
                    if col == '_':
                        width = 2
                    else:
                        max_length = max(
                            result_df[col].astype(str).apply(len).max(),
                            len(str(col))
                        )
                        width = max_length + 2
                    
                    # Convert column index to Excel column letter
                    col_letter = chr(65 + idx) if idx < 26 else chr(65 + (idx//26) - 1) + chr(65 + (idx%26))
                    worksheet.column_dimensions[col_letter].width = width
            else:
                print(f"No data processed for country {country_code}")
    
    print("\nDone!")
# I cannot put the excel file in github repo because the data is not in the public sphere yet

if __name__ == "__main__":
    base_path = "/Users/yukthabhadane/Documents/Climate Finance Thesis/Paper Alissa Jan 2025/Phase out data"
    output_path = Path(base_path).parent / "Phase_Out_Rankings_8_Countries.xlsx"
    create_ranked_phaseout_excel(base_path, output_path)

Processing 8 specified countries

Processing country: IN

Processing country: ID

Processing country: IR

Processing country: TH

Processing country: VN

Processing country: EG

Processing country: MX

Processing country: ZA

Done!
